# ประเมิน Sign2Say ด้วยวิดีโอจริง 5 คลิป
โครงทดลองที่ยังไม่มีผล: กรอก manifest แล้วรันตามลำดับ ไม่มีข้อมูลสุ่ม ไม่มีการฝึกหรือแก้ production weights

## 1. ตั้งค่า
ใช้ Python environment ที่ติดตั้ง dependencies ของ Sign2Say แล้ว ปิด LLM/TTS เพื่อวัดส่วนรู้จำคำ ไม่ใช้ clip3 ที่กันไว้ เว้นแต่ผู้ใช้อนุญาตแยกต่างหาก

In [ ]:
from pathlib import Path
import sys, json, time, hashlib
import pandas as pd
from IPython.display import display, HTML
ROOT = Path('/Users/thawaphorn/Pee4/Term1/DSI490/DSI490-Sign2Say')
EXP = Path('/Users/thawaphorn/Pee4/Term1/DSI490/การทดลอง/video_evaluation')
sys.path.insert(0, str(ROOT))
VOCAB = 'conversation'
BACKEND = 'onnx'
DEVICE = 'cpu'
RUN = EXP / 'runs' / time.strftime('%Y%m%d-%H%M%S')


## 2. กรอกข้อมูลใน manifest.json
แต่ละรายการมี `video` เป็น absolute path, `reference_words` เป็นคำภาษามือเรียงตามที่ทำจริง และ `reference_segments` เป็นรายการ `{word, start_s, end_s}` ตามคำเดียวกัน

ถ้ามีแค่คำอ้างอิง จะวัด WER/count error ได้; Segment F1 และ Recall ที่จับคู่ด้วยเวลาจะเป็นค่าว่างจนกว่าจะมี annotation เวลา ไม่ใช้ประโยคจาก LLM เป็น ground truth
ตั้ง `enabled=true` เฉพาะคลิปที่ต้องการรัน และยืนยัน `approved_for_evaluation=true` หลังตรวจว่าไม่ใช่คลิปที่ยังล็อกไว้

In [ ]:
items = json.loads((EXP / 'manifest.json').read_text())
active = [x for x in items if x['enabled']]
assert active, 'กรอก manifest.json และเปิด enabled ก่อน ไม่มีการรันคลิปอัตโนมัติ'
assert len({x['id'] for x in active}) == len(active)
for x in active:
    assert x['approved_for_evaluation'], f"ยังไม่ยืนยันคลิป {x['id']}"
    assert Path(x['video']).is_file(), x['video']
    assert x['reference_words'], 'ต้องระบุ reference_words'
    gt = x['reference_segments']
    if gt:
        assert [g['word'] for g in gt] == x['reference_words']
        assert all(0 <= g['start_s'] < g['end_s'] for g in gt)
        assert all(gt[i]['end_s'] <= gt[i+1]['start_s'] for i in range(len(gt)-1))
display(pd.DataFrame(active)[['id','video','reference_words']])

## 3. รันโมเดลจริงและบันทึกผล
โหลดโมเดลครั้งเดียว; บันทึกเวลาโหลดแยกจากเวลาประมวลผลแต่ละคลิป Latency ด้านล่างรวม pipeline และการเขียนผลของ `pipe.run` บนเครื่องนี้ ไม่ใช่ network latency
เก็บสำเนา manifest และ SHA256 ของ model artifacts เพื่อย้อนตรวจการทดลอง

In [ ]:
from modules.pipeline import SignPipeline
RUN.mkdir(parents=True, exist_ok=False)
(RUN / 'manifest.json').write_text(json.dumps(active, ensure_ascii=False, indent=2))
def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()
provenance = {'backend': BACKEND, 'device': DEVICE, 'vocab': VOCAB,
              'models': {str(f.relative_to(ROOT)):sha256_file(f) for f in (ROOT/'models').rglob('*') if f.is_file()}}
t0 = time.perf_counter()
pipe = SignPipeline(backend=BACKEND, device=DEVICE, vocab=VOCAB, use_llm=False, use_face=True)
provenance['model_load_s'] = time.perf_counter()-t0
(RUN/'provenance.json').write_text(json.dumps(provenance, indent=2))
results = {}
for x in active:
    start = time.perf_counter()
    result = pipe.run(Path(x['video']), RUN/x['id'], tts=False)
    elapsed = time.perf_counter()-start
    results[x['id']] = {'result':result, 'wall_s':elapsed}
    (RUN/(x['id']+'_timing.json')).write_text(json.dumps({'wall_s':elapsed}))
    print(x['id'], f'{elapsed:.3f}s')

## 4. Metrics
ใช้ `modules.decode.wer`, `sentence_metrics` และ `aggregate` ของ repo เพื่อใช้สูตรเดียวกับระบบเดิม
- WER บน nearest-word sequence โดยตัด transition ออก; รายงาน corpus WER เพิ่มโดยถ่วงจำนวนคำอ้างอิง
- Segment F1 ใช้ greedy one-to-one IoU ≥ 0.3 แบบเดิม; แปลงวินาทีเป็นหน่วยเฟรมของผลลัพธ์
- Recall@1/5 แบบ IoU-matched รวมคำอ้างอิงทุกคำ; ไม่ใช่ bag-of-words recall ใน `score_video`
- Accepted correctness ใช้การจับคู่เวลาเดียวกัน มีทั้งตัวเศษและตัวส่วน; ไม่มี accepted ให้เป็น N/A
- Count error = |จำนวนช่วงที่ทาย − จำนวนคำจริง|
ชื่อคำต้องใช้ canonical form เดียวกับโมเดล ตรวจ synonym ก่อนรัน ห้ามแก้คำอ้างอิงเพื่อให้ตรง prediction หลังเห็นผล

In [ ]:
from modules.decode import wer, sentence_metrics, aggregate
rows, matched_rows = [], []
for x in active:
    r = results[x['id']]['result']
    segs = [s for s in r['segments'] if s['status'] != 'transition']
    ref = x['reference_words']
    row = {'id':x['id'], 'WER':float(wer(ref,[s['nearest'] for s in segs])),
           'n_ref':len(ref), 'count_error':abs(len(segs)-len(ref)),
           'latency_s':results[x['id']]['wall_s'], 'Segment_F1':None,
           'Recall@1':None, 'Recall@5':None, 'accepted_correct':None,
           'accepted_total':sum(s['status']=='accepted' for s in segs)}
    if x['reference_segments']:
        fps = r['fps']
        gt = [(g['start_s']*fps,g['end_s']*fps) for g in x['reference_segments']]
        m = sentence_metrics(segs,gt,ref)
        matched_rows.append(m)
        row.update({'Segment_F1':m['seg_f1'], 'Recall@1':m['hit1']/m['n_inv'],
                    'Recall@5':m['hit5']/m['n_inv'],
                    'accepted_correct':sum(ok for ok,s in zip(m['correct'],segs) if s['status']=='accepted')})
    rows.append(row)
metrics = pd.DataFrame(rows)
display(metrics)
summary = {'videos':len(rows), 'corpus_WER':sum(r['WER']*r['n_ref'] for r in rows)/sum(r['n_ref'] for r in rows),
           'mean_count_error':float(metrics.count_error.mean()),
           'mean_latency_s':float(metrics.latency_s.mean()), 'timed_annotation_videos':len(matched_rows)}
if matched_rows:
    summary['repo_aggregate_timed_subset'] = aggregate(matched_rows)
(RUN/'metrics.json').write_text(json.dumps({'per_video':rows,'summary':summary},ensure_ascii=False,indent=2))
display(summary)

## 5. ภาพเฟรมและ Top-5 ต่อช่วง
แสดงเฟรมกลางช่วงและเวลาวินาที พร้อม p_correct ของช่วงนั้น
`candidate.score` เป็นคะแนน matching ไม่ใช่ probability รายคำ; คงลำดับ candidates จากระบบซึ่งอาจใช้ temporal support ร่วมด้วย ไม่เรียง cosine ใหม่ ไม่สร้าง p รายอันดับขึ้นเอง

In [ ]:
import cv2, base64, html
for x in active:
    cap = cv2.VideoCapture(x['video'])
    cards = []
    try:
        for s in results[x['id']]['result']['segments']:
            cap.set(cv2.CAP_PROP_POS_MSEC, 1000*(s['t0']+s['t1'])/2)
            ok, frame = cap.read()
            img = ''
            if ok:
                ok, data = cv2.imencode('.jpg',frame)
                if ok:
                    img = '<img style="width:240px" src="data:image/jpeg;base64,'+base64.b64encode(data).decode()+'">'
            lines = ''.join('<tr><td>'+html.escape(c['word'])+'</td><td>'+str(c.get('score'))+'</td><td>'+str(c.get('support'))+'</td></tr>' for c in s['candidates'][:5])
            cards.append(f"<div style='padding:12px;border:1px solid #bbb'><b>{s['t0']:.2f}–{s['t1']:.2f}s</b><p>p_correct={s.get('p_correct')} · {html.escape(s['status'])}</p>{img}<table><tr><th>คำ</th><th>score</th><th>support</th></tr>{lines}</table></div>")
    finally:
        cap.release()
    page = '<meta charset="utf-8"><h2>'+html.escape(x['id'])+'</h2><div style="display:flex;flex-wrap:wrap">'+''.join(cards)+'</div>'
    (RUN/(x['id']+'_visualization.html')).write_text(page)
    display(HTML(page))

## 6. สรุปการทดลองหลังรัน
- ใช้คลิปจริงกี่คลิป และมีกี่คลิปที่มี annotation เวลา?
- คำใดผิดและทายเป็นอะไร? มือหาย/ตัดช่วงผิด/จัดอันดับผิด?
- ค่า accepted correctness ต้องรายงานพร้อมจำนวน accepted และจำนวน prediction ทั้งหมด
- 5 คลิปเป็นการทดสอบเบื้องต้น ไม่แทนความแม่นยำทั่วไป
- ยังไม่มี baseline รอบก่อน → ห้ามกล่าวว่าดีขึ้น; เก็บ config และใช้คลิปชุดเดียวกันเมื่อเปรียบเทียบ

**สถานะเริ่มต้น: ยังไม่ได้รัน inference เพราะยังไม่มีวิดีโอและ ground truth ที่ผู้ใช้กำหนด**